# nb_transform: bronze → silver → gold

Runs on the `sparksmall` pool (auto-pauses after 5 min). Called by Data Factory `pl_daily` (activity `transform`).

- Parses new/changed bill-history XML and fiscal-note HTML from `bronze` with the project's own parsers (`txfr`, shipped as a zip).
- MERGEs into Delta tables in `silver` (business keys: session + bill [+ version]); unparseable files go to `silver/quarantine`.
- Rebuilds `gold` Parquet (flattened, typed) for the Azure SQL load and serverless SQL views.

Source and tests: `src/txfr/spark_jobs.py`, `tests/test_spark_jobs.py`.

In [ ]:
# parameters (Data Factory overrides these)
full = False          # re-parse every bronze file
maintenance = False   # weekly: OPTIMIZE + VACUUM silver Delta tables


In [ ]:
import json
from notebookutils import mssparkutils

LAKE = "${LAKE_ACCOUNT}"   # filled in by synapse/deploy.sh
lake = {z: f"abfss://{z}@{LAKE}.dfs.core.windows.net" for z in ("bronze", "silver", "gold")}

# Ship the project's parser package to the driver and executors (no pool-level package install needed)
sc.addPyFile(f"abfss://synapse@{LAKE}.dfs.core.windows.net/libs/txfr.zip")
from txfr import spark_jobs

In [ ]:
summary = spark_jobs.run(spark, lake, fmt="delta", full=bool(full), maintenance=bool(maintenance))

In [ ]:
# Returned to Data Factory as the activity output (status.Output.result.exitValue)
mssparkutils.notebook.exit(json.dumps(summary, default=str))